# GPT reproduction first model

 1. Read the input file
 1. Create the character-level vocabulary.
 1. Create decode and encode function

In [32]:
# 1. Read the input file.
training_data = "None"
with open("karpathy/ng-video-lecture-master/input.txt", encoding="utf-8") as f:
    training_data = f.read()
print(training_data[:100])

# 2. Create the character-level vocabulary.

class Vocabulary:
    def __init__(self,data):
        self.vocabulary = sorted(list(set(list(training_data))))
        self.stoi = { c:i for i,c in enumerate(self.vocabulary) }
        self.itos = { i:c for i,c in enumerate(self.vocabulary) }

# 3. Create decode and encode function
    def encode_data(self,input:str) -> list:
        return [self.stoi[x] for x in input ]

    def decode_data(self,input:list) -> str:
        return "".join([self.itos[x] for x in input ])

    def size(self) -> int:
        return len(self.vocabulary)


vocabulary = Vocabulary(training_data)
print(vocabulary.encode_data("hii there"))
print(vocabulary.decode_data(vocabulary.encode_data("hii there")))
# ----- PARAMETERS -----
EMBEDDING_DIM = 3


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


Ez a model most reprodukálja az lehetö legegyszerübb nyelvi modelt. Csak a loss kiszámítását tartalmazza, és a következö token generátor funkcióját

In [34]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)


class LLM_Forward_Loss_Generator(nn.Module):  # What is the torch.nn.Module?

    def __init__(self, vocabulary: Vocabulary, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_embedding_table = nn.Embedding(vocabulary.size(), EMBEDDING_DIM)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


network = LLM_Forward_Loss_Generator(vocabulary)
input = torch.tensor([vocabulary.encode_data("alma"),vocabulary.encode_data("kore")], dtype=torch.long)

prediction, loss = network(input, None)

print(network.token_embedding_table(input).shape)
print(network.token_embedding_table(input))

torch.Size([2, 4, 3])
tensor([[[ 0.4586, -1.7662,  0.5860],
         [-0.6293,  1.1385, -0.9913],
         [ 0.1700,  1.2249, -0.2345],
         [ 0.4586, -1.7662,  0.5860]],

        [[-0.2912, -0.1140, -0.3137],
         [-0.6995, -0.8961,  0.0662],
         [ 0.0045,  2.0474, -0.1575],
         [ 1.8275,  1.3035, -0.4501]]], grad_fn=<EmbeddingBackward0>)
